In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForSpeechSeq2Seq

# 1. Define your targets
YOUR_MODEL_ID = "Sanji27/fountain_base_21ep"
NEW_TOKENIZER_ID = "csebuetnlp/banglabert" # BUET's elite Bengali Tokenizer
SAVE_DIR = "./fountain_base_bengali_tok"

print("Downloading model and new tokenizer...")

# 2. Load your model and the new BUET tokenizer
model = AutoModelForSpeechSeq2Seq.from_pretrained(
    YOUR_MODEL_ID, 
    trust_remote_code=True
)
new_tokenizer = AutoTokenizer.from_pretrained(NEW_TOKENIZER_ID)

old_vocab_size = model.config.vocab_size
new_vocab_size = len(new_tokenizer)

print(f"Old Vocab Size: {old_vocab_size:,}")
print(f"New Vocab Size: {new_vocab_size:,}")

# 3. The Surgery: Resize the model's embeddings to fit the new vocabulary
print("Resizing model embeddings...")
model.resize_token_embeddings(new_vocab_size)

# 4. CRITICAL FIX: Align the Special Tokens
# Moonshine expects BOS, EOS, and PAD. BERT uses CLS, SEP, and PAD. 
# We have to tell the model config to use the new Bengali special tokens.
model.config.pad_token_id = new_tokenizer.pad_token_id
model.config.bos_token_id = new_tokenizer.cls_token_id
model.config.eos_token_id = new_tokenizer.sep_token_id
model.config.decoder_start_token_id = new_tokenizer.cls_token_id

# 5. Save the mutated model locally
print(f"Saving swap-ready model to {SAVE_DIR}...")
model.save_pretrained(SAVE_DIR)
new_tokenizer.save_pretrained(SAVE_DIR)

print("✅ Brain transplant complete! You are ready to start training from Epoch 1.")

f:\Sanjid_2203090_Kminds\venv311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


f:\Dataset\venv311\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\User\.cache\huggingface\hub\models--Sanji27--fountain_base_21ep. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 210/210 [00:00<00:00, 6048.31it/s]
f:\Dataset\venv311\Lib\site-packages\huggingface_hub\fi

Old Vocab Size: 32,768
New Vocab Size: 32,000
Resizing model embeddings...
Saving swap-ready model to ./fountain_base_bengali_tok...


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.26it/s]

✅ Brain transplant complete! You are ready to start training from Epoch 1.


In [ ]:
"""
train_base2.py
=============
Bengali Moonshine BASE fine-tune (Brain Transplant Edition)
  - Base: Your 15-epoch trained encoder + BUET BanglaBERT Tokenizer
  - BATCH_SIZE=4, GRAD_ACCUM=8 → effective batch=32
  - LR=2e-5 
"""

import os, csv, time, random
import numpy as np
import soundfile as sf
import torch
import torch.nn as nn
from pathlib import Path
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSpeechSeq2Seq
import schedulefree

# ═══════════════════════════════════════════════════════════════════════════════
#  PATHS
# ═══════════════════════════════════════════════════════════════════════════════
TRAINING_ROOT = Path(r"D:\Dataset\Lipighor_wavs")
WAVS_DIR      = TRAINING_ROOT / "wavs_asr_chunks" / "wavs"
META_CSV      = TRAINING_ROOT / "wavs_asr_chunks" / "metadata.csv"

WORK_DIR      = Path(r"F:\Dataset\moonshine-bn-base")
SAVE_DIR      = WORK_DIR / "checkpoints"

for d in [WORK_DIR, SAVE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ═══════════════════════════════════════════════════════════════════════════════
#  CONFIG
# ═══════════════════════════════════════════════════════════════════════════════
# 🚨 Point this to the local folder or repo where you saved the transplanted model!
HF_MODEL    = "./fountain_base_bengali_tok" # OR "Sanji27/fountain_base_bengali_ready"
HF_TOKEN    = "hf_"

SAMPLE_RATE    = 16_000
MAX_AUDIO_SEC  = 30.0
MIN_AUDIO_SEC  = 4.0
MAX_TOKENS     = 190 # Lowered to 190 because your new tokenizer is 1-to-1!

BATCH_SIZE     = 4
GRAD_ACCUM     = 8
EPOCHS         = 21
LR             = 2e-4
LOG_EVERY      = 50
PATIENCE       = 4

NUM_WORKERS    = 0

FILLER_WORDS   = {"মিউজিক", "প্রশংসা"}

device   = "cuda" if torch.cuda.is_available() else "cpu"
use_bf16 = device == "cuda" and torch.cuda.is_bf16_supported()
use_fp16 = device == "cuda" and not use_bf16
dtype    = torch.bfloat16 if use_bf16 else torch.float16 if use_fp16 else torch.float32

print("=" * 60)
print(f"  Model   : {HF_MODEL}  (61.5M params)")
print(f"  Device  : {device}  |  dtype: {dtype}")
if device == "cuda":
    props = torch.cuda.get_device_properties(0)
    print(f"  GPU     : {props.name}")
    print(f"  VRAM    : {props.total_memory / 1e9:.1f} GB")
print(f"  Eff.batch: {BATCH_SIZE * GRAD_ACCUM}  (paper: 32)")
print(f"  LR      : {LR}")
print("=" * 60)


# ═══════════════════════════════════════════════════════════════════════════════
#  STEP 1 — Read metadata + write TSVs
# ═══════════════════════════════════════════════════════════════════════════════
print("\n[1/4] Reading metadata.csv ...")

rows = []
with open(META_CSV, encoding="utf-8") as f:
    for row in csv.DictReader(f):
        try:
            dur   = float(row["duration"])
            text  = row["text"].strip()
            fname = Path(row["file_name"]).name
            wav   = WAVS_DIR / fname

            words        = text.split()
            filler_count = sum(1 for w in words if w in FILLER_WORDS)
            if filler_count / max(1, len(words)) > 0.5:
                continue

            if (MIN_AUDIO_SEC <= dur <= MAX_AUDIO_SEC
                    and len(text) >= 3
                    and wav.exists()):
                rows.append({"wav": wav, "text": text, "dur": dur})
        except (ValueError, KeyError):
            pass

print(f"   ✓ {len(rows):,} valid rows (filler filtered)")

if not (WORK_DIR / "train.tsv").exists():
    random.seed(42)
    random.shuffle(rows)
    n = len(rows)
    splits = {
        "train": rows[:int(n * 0.90)],
        "dev":   rows[int(n * 0.90): int(n * 0.95)],
        "test":  rows[int(n * 0.95):],
    }
    for name, split_rows in splits.items():
        with open(WORK_DIR / f"{name}.tsv", "w", encoding="utf-8") as f:
            for r in split_rows:
                f.write(f"{r['wav']}\t{r['text']}\n")
        hrs = sum(r["dur"] for r in split_rows) / 3600
        print(f"   {name:<8} {len(split_rows):>6,} utterances   {hrs:.1f}h")
else:
    print("   ✓ TSV files already exist — skipping split")


# ═══════════════════════════════════════════════════════════════════════════════
#  STEP 2 — Load tokenizer + model
# ═══════════════════════════════════════════════════════════════════════════════
print(f"\n[2/4] Loading {HF_MODEL} ...")
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN

tokenizer = AutoTokenizer.from_pretrained(
    HF_MODEL, trust_remote_code=True
)
print(f"   ✓ Tokenizer — vocab: {tokenizer.vocab_size}")

model = AutoModelForSpeechSeq2Seq.from_pretrained(
    HF_MODEL, trust_remote_code=True, dtype=torch.float32
)

BOS_ID = model.config.decoder_start_token_id or tokenizer.cls_token_id
EOS_ID = model.config.eos_token_id or tokenizer.sep_token_id
PAD_ID = model.config.pad_token_id or tokenizer.pad_token_id or 0

print(f"   ✓ Token IDs — BOS:{BOS_ID}  EOS:{EOS_ID}  PAD:{PAD_ID}")

# Clear generation limits
model.generation_config.max_length = None
model.gradient_checkpointing_enable()

total     = sum(p.numel() for p in model.parameters()) / 1e6
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad) / 1e6
print(f"   ✓ {total:.1f}M params  {trainable:.1f}M trainable")
print(f"   ✓ Gradient checkpointing enabled")

model = model.to(device)


# ═══════════════════════════════════════════════════════════════════════════════
#  STEP 3 — Dataset + DataLoader
# ═══════════════════════════════════════════════════════════════════════════════
print("\n[3/4] Building dataloaders ...")

class BengaliASRDataset(Dataset):
    def __init__(self, tsv_path):
        self.samples = []
        with open(tsv_path, encoding="utf-8") as f:
            for line in f:
                parts = line.strip().split("\t", 1)
                if len(parts) == 2:
                    self.samples.append((Path(parts[0]), parts[1]))
        print(f"   {len(self.samples):,} samples — {Path(tsv_path).name}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        try:
            wav_path, transcript = self.samples[idx]
            audio, _ = sf.read(str(wav_path), dtype="float32", always_2d=True)
            audio = audio.mean(axis=1) # Downmix stereo safely

            dur = len(audio) / SAMPLE_RATE
            if not (MIN_AUDIO_SEC <= dur <= MAX_AUDIO_SEC):
                return None

            remainder = len(audio) % 160
            if remainder:
                audio = np.concatenate(
                    [audio, np.zeros(160 - remainder, dtype=np.float32)]
                )

            ids = tokenizer.encode(transcript, add_special_tokens=False)
            if len(ids) == 0 or len(ids) > MAX_TOKENS - 2:
                return None

            ids = [BOS_ID] + ids + [EOS_ID]
            return {
                "audio":     torch.tensor(audio, dtype=torch.float32),
                "input_ids": torch.tensor(ids,   dtype=torch.long),
            }
        except Exception:
            return None


def collate_fn(batch):
    batch = [b for b in batch if b is not None]
    if not batch: return None
    
    max_a   = ((max(b["audio"].shape[0] for b in batch) + 159) // 160) * 160
    audio_b = torch.zeros(len(batch), max_a)
    
    # 🛠️ FIX 1: Add Attention Mask for audio padding
    audio_mask = torch.zeros(len(batch), max_a, dtype=torch.long)
    
    for i, b in enumerate(batch):
        audio_b[i, :b["audio"].shape[0]] = b["audio"]
        audio_mask[i, :b["audio"].shape[0]] = 1
        
    max_t   = max(b["input_ids"].shape[0] for b in batch)
    token_b = torch.full((len(batch), max_t), -100, dtype=torch.long)
    for i, b in enumerate(batch):
        token_b[i, :b["input_ids"].shape[0]] = b["input_ids"]
        
    return {"audio": audio_b, "audio_mask": audio_mask, "input_ids": token_b}


train_ds = BengaliASRDataset(WORK_DIR / "train.tsv")
val_ds   = BengaliASRDataset(WORK_DIR / "dev.tsv")

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True,
    collate_fn=collate_fn, num_workers=NUM_WORKERS,
    pin_memory=(device == "cuda"),
)
val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False,
    collate_fn=collate_fn, num_workers=NUM_WORKERS,
)

print(f"   ✓ Train  : {len(train_loader)} batches")
print(f"   ✓ Val    : {len(val_loader)} batches")


# ═══════════════════════════════════════════════════════════════════════════════
#  STEP 4 — Train
# ═══════════════════════════════════════════════════════════════════════════════
print("\n[4/4] Training ...")

optimizer = schedulefree.AdamWScheduleFree(
    model.parameters(),
    lr=LR,
    betas=(0.9, 0.999),
    weight_decay=1e-2,
    warmup_steps=500,
)

scaler    = torch.amp.GradScaler("cuda", enabled=(device == "cuda"))
amp_dtype = dtype if device == "cuda" else torch.float32

start_epoch = 1
BEST_VAL    = float("inf")
BEST_WER    = float("inf")
patience_ct = 0

def greedy_wer(hyps, refs):
    total_w = total_e = 0
    for h, r in zip(hyps, refs):
        h, r = h.split(), r.split()
        total_w += len(r)
        d = list(range(len(r) + 1))
        for hc in h:
            p, d[0] = d[:], d[0] + 1
            for j, rc in enumerate(r):
                d[j+1] = min(p[j] + (hc != rc), d[j] + 1, p[j+1] + 1)
        total_e += d[len(r)]
    return total_e / max(1, total_w)


def train_epoch(epoch):
    optimizer.train()
    model.train()
    total_loss, t0 = 0.0, time.time()
    optimizer.zero_grad()
    steps = 0

    for step, batch in enumerate(train_loader):
        if batch is None: continue
        
        audio      = batch["audio"].to(device, non_blocking=True)
        audio_mask = batch["audio_mask"].to(device, non_blocking=True)
        input_ids  = batch["input_ids"].to(device, non_blocking=True)
        
        dec_input = input_ids[:, :-1].clone()
        labels    = input_ids[:, 1:].clone()
        dec_input[dec_input == -100] = PAD_ID

        with torch.autocast(device_type=device, dtype=amp_dtype, enabled=(device == "cuda")):
            # 🛠️ FIX 1 (applied): Pass attention_mask to model
            out = model(
                input_values=audio, 
                attention_mask=audio_mask,
                decoder_input_ids=dec_input
            )
            loss = nn.functional.cross_entropy(
                out.logits.reshape(-1, out.logits.size(-1)),
                labels.reshape(-1),
                ignore_index=-100,
            ) / GRAD_ACCUM

        scaler.scale(loss).backward()
        total_loss += loss.item() * GRAD_ACCUM

        # 🛠️ FIX 2: Handle dangling gradients at the end of the DataLoader
        is_accum_step = (step + 1) % GRAD_ACCUM == 0
        is_last_step  = (step + 1) == len(train_loader)

        if is_accum_step or is_last_step:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
            steps += 1

            if steps % LOG_EVERY == 0:
                avg = total_loss / (step + 1)
                eta = (time.time() - t0) / steps * ((len(train_loader) // GRAD_ACCUM) - steps)
                print(f"   ep{epoch} [{steps:>4}/{(len(train_loader)+GRAD_ACCUM-1)//GRAD_ACCUM}] "
                      f"loss={avg:.4f}  ETA={eta/60:.0f}min")

    return total_loss / max(1, len(train_loader))


@torch.no_grad()
def validate():
    optimizer.eval()
    model.eval()
    total_loss, n = 0.0, 0
    hyps, refs = [], []

    for batch in val_loader:
        if batch is None: continue
        audio      = batch["audio"].to(device, non_blocking=True)
        audio_mask = batch["audio_mask"].to(device, non_blocking=True)
        input_ids  = batch["input_ids"].to(device, non_blocking=True)
        
        dec_input = input_ids[:, :-1].clone()
        labels    = input_ids[:, 1:].clone()
        dec_input[dec_input == -100] = PAD_ID

        with torch.autocast(device_type=device, dtype=amp_dtype, enabled=(device == "cuda")):
            # Calculate validation loss using teacher forcing
            out = model(
                input_values=audio, 
                attention_mask=audio_mask,
                decoder_input_ids=dec_input
            )
            loss = nn.functional.cross_entropy(
                out.logits.reshape(-1, out.logits.size(-1)),
                labels.reshape(-1),
                ignore_index=-100,
            )
            total_loss += loss.item()
            n += 1

        # 🛠️ FIX 3: Calculate real WER using autoregressive generation
        if len(hyps) < 64: # Only generate a subset to keep epochs fast
            generated_ids = model.generate(
                inputs=audio,
                attention_mask=audio_mask,
                max_new_tokens=MAX_TOKENS,
                pad_token_id=PAD_ID,
                eos_token_id=EOS_ID
            )
            for g_ids, ref_ids in zip(generated_ids, input_ids):
                # Strip out padding from references
                ref_ids = ref_ids[ref_ids != -100].tolist()
                hyps.append(tokenizer.decode(g_ids.tolist(), skip_special_tokens=True))
                refs.append(tokenizer.decode(ref_ids, skip_special_tokens=True))

    return total_loss / max(1, n), greedy_wer(hyps, refs)


def save_checkpoint(epoch, val_loss, wer):
    best_dir = SAVE_DIR / "best"
    best_dir.mkdir(exist_ok=True)
    optimizer.eval()
    model.save_pretrained(best_dir)
    tokenizer.save_pretrained(best_dir)
    torch.save(model.state_dict(), best_dir / "model_state.pt")
    torch.save({
        "epoch":    epoch,
        "val_loss": val_loss,
        "wer":      wer,
        "patience": patience_ct,
        "optimizer": optimizer.state_dict(),
    }, best_dir / "training_state.pt")
    optimizer.train()
    print(f"   ✅ Saved → epoch={epoch}  "
          f"val_loss={val_loss:.4f}  WER={wer*100:.1f}%")


print(f"\n{'='*60}")
print(f"  Starting Training Loop")
print(f"{'='*60}\n")

for epoch in range(start_epoch, EPOCHS + 1):
    t0         = time.time()
    train_loss = train_epoch(epoch)
    val_loss, wer = validate()
    epoch_min  = (time.time() - t0) / 60

    print(f"\nepoch {epoch}/{EPOCHS}  "
          f"train={train_loss:.4f}  val={val_loss:.4f}  "
          f"WER={wer*100:.1f}%  time={epoch_min:.0f}min")

    if val_loss < BEST_VAL:
        BEST_VAL    = val_loss
        BEST_WER    = wer
        patience_ct = 0
        save_checkpoint(epoch, val_loss, wer)
    else:
        patience_ct += 1
        print(f"   ⚠️  No improvement — patience {patience_ct}/{PATIENCE}")
        if patience_ct >= PATIENCE:
            print(f"\n🛑 Early stopping at epoch {epoch}")
            break
    print()

print(f"\n{'='*60}")
print(f"  Training complete!")
print(f"  Best val_loss : {BEST_VAL:.4f}")
print(f"  Best WER      : {BEST_WER*100:.1f}%")
print(f"{'='*60}")

f:\Dataset\venv311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


  Model   : ./fountain_base_bengali_tok  (61.5M params)
  Device  : cuda  |  dtype: torch.bfloat16
  GPU     : NVIDIA GeForce RTX 4070
  VRAM    : 12.9 GB
  Eff.batch: 32  (paper: 32)
  LR      : 0.0002

[1/4] Reading metadata.csv ...
   ✓ 118,616 valid rows (filler filtered)
   ✓ TSV files already exist — skipping split

[2/4] Loading ./fountain_base_bengali_tok ...
   ✓ Tokenizer — vocab: 32000


Loading weights: 100%|██████████| 210/210 [00:00<00:00, 372.29it/s]


   ✓ Token IDs — BOS:2  EOS:3  PAD:0
   ✓ 61.2M params  61.2M trainable
   ✓ Gradient checkpointing enabled

[3/4] Building dataloaders ...
   106,754 samples — train.tsv
   5,931 samples — dev.tsv
   ✓ Train  : 26689 batches
   ✓ Val    : 1483 batches

[4/4] Training ...

  Starting Training Loop



[transformers] `use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


...

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.45it/s]


   ✅ Saved → epoch=6  val_loss=0.9042  WER=100.0%

   ep7 [  50/3337] loss=0.4915  ETA=146min
   ep7 [ 100/3337] loss=0.4938  ETA=145min
   ep7 [ 150/3337] loss=0.4899  ETA=142min
   ep7 [ 200/3337] loss=0.4866  ETA=140min
   ep7 [ 250/3337] loss=0.4849  ETA=138min
   ep7 [ 300/3337] loss=0.4838  ETA=135min
   ep7 [ 350/3337] loss=0.4835  ETA=133min
   ep7 [ 400/3337] loss=0.4837  ETA=131min
   ep7 [ 450/3337] loss=0.4849  ETA=129min
   ep7 [ 500/3337] loss=0.4836  ETA=127min
   ep7 [ 550/3337] loss=0.4842  ETA=125min
   ep7 [ 600/3337] loss=0.4843  ETA=123min
   ep7 [ 650/3337] loss=0.4833  ETA=120min
   ep7 [ 700/3337] loss=0.4838  ETA=118min
   ep7 [ 750/3337] loss=0.4837  ETA=116min
   ep7 [ 800/3337] loss=0.4823  ETA=114min
   ep7 [ 850/3337] loss=0.4811  ETA=112min
   ep7 [ 900/3337] loss=0.4814  ETA=110min
   ep7 [ 950/3337] loss=0.4819  ETA=107min
   ep7 [1000/3337] loss=0.4823  ETA=105min
   ep7 [1050/3337] loss=0.4813  ETA=103min
   ep7 [1100/3337] loss=0.4807  ETA=101min
   

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.25it/s]


   ✅ Saved → epoch=7  val_loss=0.8983  WER=100.0%

   ep8 [  50/3337] loss=0.3988  ETA=146min
   ep8 [ 100/3337] loss=0.4045  ETA=143min
   ep8 [ 150/3337] loss=0.4056  ETA=142min
   ep8 [ 200/3337] loss=0.4106  ETA=139min
   ep8 [ 250/3337] loss=0.4091  ETA=137min
   ep8 [ 300/3337] loss=0.4096  ETA=135min
   ep8 [ 350/3337] loss=0.4078  ETA=133min


KeyboardInterrupt: 

In [2]:
"""
train_base3.py
=============
Bengali Moonshine BASE fine-tune — FIXED Edition
  Fixes applied vs train_base2.py:
    [1] LR corrected: 2e-4 → 2e-5
    [2] decoder_start_token_id added to model.generate() in validate()
    [3] Checkpoint saved on best WER, not just best val_loss
    [4] WER subset randomly sampled each epoch (not always first 64)
    [5] librosa resampling added to Dataset (force 16kHz)
    [6] Resume from checkpoint support added
    [7] ETA calculation corrected (ceiling division)
    [8] BOS_ID/EOS_ID passed explicitly into Dataset (no global dependency)
"""

import os, csv, time, random
import numpy as np
import librosa
import torch
import torch.nn as nn
from pathlib import Path
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSpeechSeq2Seq
import schedulefree

# ═══════════════════════════════════════════════════════════════════════════════
#  PATHS
# ═══════════════════════════════════════════════════════════════════════════════
TRAINING_ROOT = Path(r"D:\Dataset\Lipighor_wavs")
WAVS_DIR      = TRAINING_ROOT / "wavs_asr_chunks" / "wavs"
META_CSV      = TRAINING_ROOT / "wavs_asr_chunks" / "metadata.csv"

WORK_DIR      = Path(r"F:\Dataset\moonshine-bn-base")
SAVE_DIR      = WORK_DIR / "checkpoints"

for d in [WORK_DIR, SAVE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ═══════════════════════════════════════════════════════════════════════════════
#  CONFIG
# ═══════════════════════════════════════════════════════════════════════════════
HF_MODEL    = "./fountain_base_bengali_tok"
HF_TOKEN    = "hf_"

SAMPLE_RATE    = 16_000
MAX_AUDIO_SEC  = 30.0
MIN_AUDIO_SEC  = 4.0
MAX_TOKENS     = 190

BATCH_SIZE     = 4
GRAD_ACCUM     = 8
EPOCHS         = 21
LR             = 2e-5          # ✅ FIX 1: was 2e-4
LOG_EVERY      = 50
PATIENCE       = 4
WER_SUBSET     = 64            # How many val samples to use for WER each epoch

NUM_WORKERS    = 0

# ─── Resume ──────────────────────────────────────────────────────────────────
RESUME         = True          # Set False to train from scratch
RESUME_DIR     = SAVE_DIR / "best"

FILLER_WORDS   = {"মিউজিক", "প্রশংসা"}

device   = "cuda" if torch.cuda.is_available() else "cpu"
use_bf16 = device == "cuda" and torch.cuda.is_bf16_supported()
use_fp16 = device == "cuda" and not use_bf16
dtype    = torch.bfloat16 if use_bf16 else torch.float16 if use_fp16 else torch.float32

print("=" * 60)
print(f"  Model   : {HF_MODEL}  (61.5M params)")
print(f"  Device  : {device}  |  dtype: {dtype}")
if device == "cuda":
    props = torch.cuda.get_device_properties(0)
    print(f"  GPU     : {props.name}")
    print(f"  VRAM    : {props.total_memory / 1e9:.1f} GB")
print(f"  Eff.batch: {BATCH_SIZE * GRAD_ACCUM}")
print(f"  LR      : {LR}")
print(f"  Resume  : {RESUME}")
print("=" * 60)


# ═══════════════════════════════════════════════════════════════════════════════
#  STEP 1 — Read metadata + write TSVs
# ═══════════════════════════════════════════════════════════════════════════════
print("\n[1/4] Reading metadata.csv ...")

rows = []
with open(META_CSV, encoding="utf-8") as f:
    for row in csv.DictReader(f):
        try:
            dur   = float(row["duration"])
            text  = row["text"].strip()
            fname = Path(row["file_name"]).name
            wav   = WAVS_DIR / fname

            words        = text.split()
            filler_count = sum(1 for w in words if w in FILLER_WORDS)
            if filler_count / max(1, len(words)) > 0.5:
                continue

            if (MIN_AUDIO_SEC <= dur <= MAX_AUDIO_SEC
                    and len(text) >= 3
                    and wav.exists()):
                rows.append({"wav": wav, "text": text, "dur": dur})
        except (ValueError, KeyError):
            pass

print(f"   ✓ {len(rows):,} valid rows (filler filtered)")

if not (WORK_DIR / "train.tsv").exists():
    random.seed(42)
    random.shuffle(rows)
    n = len(rows)
    splits = {
        "train": rows[:int(n * 0.90)],
        "dev":   rows[int(n * 0.90): int(n * 0.95)],
        "test":  rows[int(n * 0.95):],
    }
    for name, split_rows in splits.items():
        with open(WORK_DIR / f"{name}.tsv", "w", encoding="utf-8") as f:
            for r in split_rows:
                f.write(f"{r['wav']}\t{r['text']}\n")
        hrs = sum(r["dur"] for r in split_rows) / 3600
        print(f"   {name:<8} {len(split_rows):>6,} utterances   {hrs:.1f}h")
else:
    print("   ✓ TSV files already exist — skipping split")


# ═══════════════════════════════════════════════════════════════════════════════
#  STEP 2 — Load tokenizer + model
# ═══════════════════════════════════════════════════════════════════════════════
print(f"\n[2/4] Loading {HF_MODEL} ...")
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN

tokenizer = AutoTokenizer.from_pretrained(HF_MODEL, trust_remote_code=True)
print(f"   ✓ Tokenizer — vocab: {tokenizer.vocab_size}")

model = AutoModelForSpeechSeq2Seq.from_pretrained(
    HF_MODEL, trust_remote_code=True, dtype=torch.float32
)

# ✅ FIX 8: Resolve IDs once here, pass explicitly — no global dependency later
BOS_ID = model.config.decoder_start_token_id or tokenizer.cls_token_id
EOS_ID = model.config.eos_token_id            or tokenizer.sep_token_id
PAD_ID = model.config.pad_token_id            or tokenizer.pad_token_id or 0

print(f"   ✓ Token IDs — BOS:{BOS_ID}  EOS:{EOS_ID}  PAD:{PAD_ID}")

model.generation_config.max_length = None
model.gradient_checkpointing_enable()

total     = sum(p.numel() for p in model.parameters()) / 1e6
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad) / 1e6
print(f"   ✓ {total:.1f}M params  {trainable:.1f}M trainable")
print(f"   ✓ Gradient checkpointing enabled")

model = model.to(device)


# ═══════════════════════════════════════════════════════════════════════════════
#  STEP 3 — Dataset + DataLoader
# ═══════════════════════════════════════════════════════════════════════════════
print("\n[3/4] Building dataloaders ...")

class BengaliASRDataset(Dataset):
    def __init__(self, tsv_path, bos_id, eos_id):
        # ✅ FIX 8: receive IDs explicitly, no global scope dependency
        self.bos_id  = bos_id
        self.eos_id  = eos_id
        self.samples = []
        with open(tsv_path, encoding="utf-8") as f:
            for line in f:
                parts = line.strip().split("\t", 1)
                if len(parts) == 2:
                    self.samples.append((Path(parts[0]), parts[1]))
        print(f"   {len(self.samples):,} samples — {Path(tsv_path).name}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        try:
            wav_path, transcript = self.samples[idx]

            # ✅ FIX 5: Force 16kHz resample via librosa (was sf.read with no resample)
            audio, _ = librosa.load(str(wav_path), sr=SAMPLE_RATE, mono=True)
            audio    = audio.astype(np.float32)

            dur = len(audio) / SAMPLE_RATE
            if not (MIN_AUDIO_SEC <= dur <= MAX_AUDIO_SEC):
                return None

            remainder = len(audio) % 160
            if remainder:
                audio = np.concatenate(
                    [audio, np.zeros(160 - remainder, dtype=np.float32)]
                )

            ids = tokenizer.encode(transcript, add_special_tokens=False)
            if len(ids) == 0 or len(ids) > MAX_TOKENS - 2:
                return None

            ids = [self.bos_id] + ids + [self.eos_id]
            return {
                "audio":     torch.tensor(audio, dtype=torch.float32),
                "input_ids": torch.tensor(ids,   dtype=torch.long),
            }
        except Exception:
            return None


def collate_fn(batch):
    batch = [b for b in batch if b is not None]
    if not batch:
        return None

    max_a      = ((max(b["audio"].shape[0] for b in batch) + 159) // 160) * 160
    audio_b    = torch.zeros(len(batch), max_a)
    audio_mask = torch.zeros(len(batch), max_a, dtype=torch.long)

    for i, b in enumerate(batch):
        audio_b[i, :b["audio"].shape[0]]    = b["audio"]
        audio_mask[i, :b["audio"].shape[0]] = 1

    max_t   = max(b["input_ids"].shape[0] for b in batch)
    token_b = torch.full((len(batch), max_t), -100, dtype=torch.long)
    for i, b in enumerate(batch):
        token_b[i, :b["input_ids"].shape[0]] = b["input_ids"]

    return {"audio": audio_b, "audio_mask": audio_mask, "input_ids": token_b}


train_ds = BengaliASRDataset(WORK_DIR / "train.tsv", BOS_ID, EOS_ID)
val_ds   = BengaliASRDataset(WORK_DIR / "dev.tsv",   BOS_ID, EOS_ID)

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True,
    collate_fn=collate_fn, num_workers=NUM_WORKERS,
    pin_memory=(device == "cuda"),
)
val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=True,   # ✅ FIX 4: shuffle=True so WER subset is random
    collate_fn=collate_fn, num_workers=NUM_WORKERS,
)

print(f"   ✓ Train  : {len(train_loader)} batches")
print(f"   ✓ Val    : {len(val_loader)} batches")


# ═══════════════════════════════════════════════════════════════════════════════
#  STEP 4 — Train
# ═══════════════════════════════════════════════════════════════════════════════
print("\n[4/4] Training ...")

optimizer = schedulefree.AdamWScheduleFree(
    model.parameters(),
    lr=LR,
    betas=(0.9, 0.999),
    weight_decay=1e-2,
    warmup_steps=500,
)

scaler    = torch.amp.GradScaler("cuda", enabled=(device == "cuda"))
amp_dtype = dtype if device == "cuda" else torch.float32

# ─── Defaults (overwritten if resuming) ──────────────────────────────────────
start_epoch = 1
BEST_VAL    = float("inf")
BEST_WER    = float("inf")
patience_ct = 0

# ─── Resume ──────────────────────────────────────────────────────────────────
if RESUME and (RESUME_DIR / "training_state.pt").exists():
    print(f"\n   Resuming from: {RESUME_DIR}")
    state = torch.load(RESUME_DIR / "training_state.pt", map_location=device)
    model.load_state_dict(
        torch.load(RESUME_DIR / "model_state.pt", map_location=device)
    )
    optimizer.load_state_dict(state["optimizer"])

    # ✅ Override LR without losing momentum/adaptive terms
    for pg in optimizer.param_groups:
        pg["lr"] = LR

    start_epoch = state["epoch"] + 1
    BEST_VAL    = state["val_loss"]
    BEST_WER    = state.get("wer", float("inf"))
    patience_ct = state.get("patience", 0)

    print(f"   ✓ Resumed  epoch={state['epoch']}  "
          f"val_loss={BEST_VAL:.4f}  WER={BEST_WER*100:.1f}%  "
          f"patience={patience_ct}/{PATIENCE}")
    print(f"   ✓ LR overridden → {LR}")
else:
    print("   Starting from scratch ...")


# ─── Helpers ─────────────────────────────────────────────────────────────────
def greedy_wer(hyps, refs):
    total_w = total_e = 0
    for h, r in zip(hyps, refs):
        h, r = h.split(), r.split()
        total_w += len(r)
        d = list(range(len(r) + 1))
        for hc in h:
            p, d[0] = d[:], d[0] + 1
            for j, rc in enumerate(r):
                d[j+1] = min(p[j] + (hc != rc), d[j] + 1, p[j+1] + 1)
        total_e += d[len(r)]
    return total_e / max(1, total_w)


def train_epoch(epoch):
    optimizer.train()
    model.train()
    total_loss, t0 = 0.0, time.time()
    optimizer.zero_grad()
    steps      = 0
    total_steps = (len(train_loader) + GRAD_ACCUM - 1) // GRAD_ACCUM  # ✅ FIX 7

    for step, batch in enumerate(train_loader):
        if batch is None:
            continue

        audio      = batch["audio"].to(device, non_blocking=True)
        audio_mask = batch["audio_mask"].to(device, non_blocking=True)
        input_ids  = batch["input_ids"].to(device, non_blocking=True)

        dec_input = input_ids[:, :-1].clone()
        labels    = input_ids[:, 1:].clone()
        dec_input[dec_input == -100] = PAD_ID

        with torch.autocast(device_type=device, dtype=amp_dtype, enabled=(device == "cuda")):
            out = model(
                input_values=audio,
                attention_mask=audio_mask,
                decoder_input_ids=dec_input,
            )
            loss = nn.functional.cross_entropy(
                out.logits.reshape(-1, out.logits.size(-1)),
                labels.reshape(-1),
                ignore_index=-100,
            ) / GRAD_ACCUM

        scaler.scale(loss).backward()
        total_loss += loss.item() * GRAD_ACCUM

        is_accum_step = (step + 1) % GRAD_ACCUM == 0
        is_last_step  = (step + 1) == len(train_loader)

        if is_accum_step or is_last_step:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
            steps += 1

            if steps % LOG_EVERY == 0:
                avg = total_loss / (step + 1)
                # ✅ FIX 7: ceiling division for correct remaining step count
                remaining = total_steps - steps
                eta = (time.time() - t0) / steps * remaining
                print(f"   ep{epoch} [{steps:>4}/{total_steps}] "
                      f"loss={avg:.4f}  ETA={eta/60:.0f}min")

    return total_loss / max(1, len(train_loader))


@torch.no_grad()
def validate():
    optimizer.eval()
    model.eval()
    total_loss, n = 0.0, 0
    hyps, refs    = [], []

    for batch in val_loader:
        if batch is None:
            continue

        audio      = batch["audio"].to(device, non_blocking=True)
        audio_mask = batch["audio_mask"].to(device, non_blocking=True)
        input_ids  = batch["input_ids"].to(device, non_blocking=True)

        dec_input = input_ids[:, :-1].clone()
        labels    = input_ids[:, 1:].clone()
        dec_input[dec_input == -100] = PAD_ID

        with torch.autocast(device_type=device, dtype=amp_dtype, enabled=(device == "cuda")):
            out = model(
                input_values=audio,
                attention_mask=audio_mask,
                decoder_input_ids=dec_input,
            )
            loss = nn.functional.cross_entropy(
                out.logits.reshape(-1, out.logits.size(-1)),
                labels.reshape(-1),
                ignore_index=-100,
            )
            total_loss += loss.item()
            n += 1

        # ✅ FIX 2: decoder_start_token_id added
        # ✅ FIX 4: val_loader shuffled → this subset is random each epoch
        if len(hyps) < WER_SUBSET:
            generated_ids = model.generate(
                inputs=audio,
                attention_mask=audio_mask,
                max_new_tokens=MAX_TOKENS,
                pad_token_id=PAD_ID,
                eos_token_id=EOS_ID,
                decoder_start_token_id=BOS_ID,   # ✅ FIX 2
                num_beams=4,
                no_repeat_ngram_size=3,
                repetition_penalty=1.2,
            )
            for g_ids, ref_ids in zip(generated_ids, input_ids):
                ref_ids = ref_ids[ref_ids != -100].tolist()
                hyps.append(tokenizer.decode(g_ids.tolist(), skip_special_tokens=True))
                refs.append(tokenizer.decode(ref_ids,        skip_special_tokens=True))

    return total_loss / max(1, n), greedy_wer(hyps, refs)


def save_checkpoint(epoch, val_loss, wer):
    best_dir = SAVE_DIR / "best"
    best_dir.mkdir(exist_ok=True)
    optimizer.eval()
    model.save_pretrained(best_dir)
    tokenizer.save_pretrained(best_dir)
    torch.save(model.state_dict(), best_dir / "model_state.pt")
    torch.save({
        "epoch":     epoch,
        "val_loss":  val_loss,
        "wer":       wer,
        "patience":  patience_ct,
        "optimizer": optimizer.state_dict(),
    }, best_dir / "training_state.pt")
    optimizer.train()
    print(f"   ✅ Saved → epoch={epoch}  "
          f"val_loss={val_loss:.4f}  WER={wer*100:.1f}%")


# ─── Training Loop ───────────────────────────────────────────────────────────
print(f"\n{'='*60}")
print(f"  Starting Training Loop  (epoch {start_epoch} → {EPOCHS})")
print(f"{'='*60}\n")

for epoch in range(start_epoch, EPOCHS + 1):
    t0            = time.time()
    train_loss    = train_epoch(epoch)
    val_loss, wer = validate()
    epoch_min     = (time.time() - t0) / 60

    print(f"\nepoch {epoch}/{EPOCHS}  "
          f"train={train_loss:.4f}  val={val_loss:.4f}  "
          f"WER={wer*100:.1f}%  time={epoch_min:.0f}min")

    # ✅ FIX 3: Save on best WER (not just best val_loss)
    if wer < BEST_WER:
        BEST_VAL    = val_loss
        BEST_WER    = wer
        patience_ct = 0
        save_checkpoint(epoch, val_loss, wer)
    else:
        patience_ct += 1
        print(f"   ⚠️  No WER improvement — patience {patience_ct}/{PATIENCE}")
        if patience_ct >= PATIENCE:
            print(f"\n🛑 Early stopping at epoch {epoch}")
            break
    print()

print(f"\n{'='*60}")
print(f"  Training complete!")
print(f"  Best val_loss : {BEST_VAL:.4f}")
print(f"  Best WER      : {BEST_WER*100:.1f}%")
print(f"{'='*60}")

  Model   : ./fountain_base_bengali_tok  (61.5M params)
  Device  : cuda  |  dtype: torch.bfloat16
  GPU     : NVIDIA GeForce RTX 4070
  VRAM    : 12.9 GB
  Eff.batch: 32
  LR      : 2e-05
  Resume  : True

[1/4] Reading metadata.csv ...
   ✓ 118,616 valid rows (filler filtered)
   ✓ TSV files already exist — skipping split

[2/4] Loading ./fountain_base_bengali_tok ...
   ✓ Tokenizer — vocab: 32000


Loading weights: 100%|██████████| 210/210 [00:00<00:00, 287.62it/s]


   ✓ Token IDs — BOS:2  EOS:3  PAD:0
   ✓ 61.2M params  61.2M trainable
   ✓ Gradient checkpointing enabled

[3/4] Building dataloaders ...
   106,754 samples — train.tsv
   5,931 samples — dev.tsv
   ✓ Train  : 26689 batches
   ✓ Val    : 1483 batches

[4/4] Training ...

   Resuming from: F:\Dataset\moonshine-bn-base\checkpoints\best
   ✓ Resumed  epoch=7  val_loss=0.8983  WER=100.0%  patience=0/4
   ✓ LR overridden → 2e-05

  Starting Training Loop  (epoch 8 → 21)

   ep8 [  50/3337] loss=0.4023  ETA=153min
   ep8 [ 100/3337] loss=0.4068  ETA=147min
   ep8 [ 150/3337] loss=0.4032  ETA=143min
   ep8 [ 200/3337] loss=0.4012  ETA=140min
   ep8 [ 250/3337] loss=0.4021  ETA=138min
   ep8 [ 300/3337] loss=0.4016  ETA=135min
   ep8 [ 350/3337] loss=0.4004  ETA=133min
   ep8 [ 400/3337] loss=0.3995  ETA=130min
   ep8 [ 450/3337] loss=0.4004  ETA=128min
   ep8 [ 500/3337] loss=0.3986  ETA=126min
   ep8 [ 550/3337] loss=0.3994  ETA=124min
   ep8 [ 600/3337] loss=0.3990  ETA=121min
   ep8 [ 65

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.41it/s]


   ✅ Saved → epoch=8  val_loss=0.9184  WER=22.3%

   ep9 [  50/3337] loss=0.3513  ETA=147min
   ep9 [ 100/3337] loss=0.3520  ETA=145min
   ep9 [ 150/3337] loss=0.3553  ETA=142min
   ep9 [ 200/3337] loss=0.3569  ETA=140min
   ep9 [ 250/3337] loss=0.3584  ETA=138min
   ep9 [ 300/3337] loss=0.3560  ETA=136min
   ep9 [ 350/3337] loss=0.3557  ETA=136min
   ep9 [ 400/3337] loss=0.3559  ETA=134min
   ep9 [ 450/3337] loss=0.3571  ETA=131min
   ep9 [ 500/3337] loss=0.3570  ETA=128min
   ep9 [ 550/3337] loss=0.3557  ETA=125min
   ep9 [ 600/3337] loss=0.3585  ETA=123min
   ep9 [ 650/3337] loss=0.3581  ETA=120min
   ep9 [ 700/3337] loss=0.3574  ETA=117min
   ep9 [ 750/3337] loss=0.3568  ETA=115min
   ep9 [ 800/3337] loss=0.3565  ETA=113min
   ep9 [ 850/3337] loss=0.3557  ETA=110min
   ep9 [ 900/3337] loss=0.3549  ETA=108min
   ep9 [ 950/3337] loss=0.3557  ETA=105min
   ep9 [1000/3337] loss=0.3546  ETA=104min
   ep9 [1050/3337] loss=0.3546  ETA=102min
   ep9 [1100/3337] loss=0.3543  ETA=99min
   ep

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.28it/s]


   ✅ Saved → epoch=9  val_loss=0.9412  WER=20.4%

   ep10 [  50/3337] loss=0.3288  ETA=165min
   ep10 [ 100/3337] loss=0.3299  ETA=163min
   ep10 [ 150/3337] loss=0.3310  ETA=160min
   ep10 [ 200/3337] loss=0.3379  ETA=158min
   ep10 [ 250/3337] loss=0.3410  ETA=155min
   ep10 [ 300/3337] loss=0.3404  ETA=153min
   ep10 [ 350/3337] loss=0.3397  ETA=151min
   ep10 [ 400/3337] loss=0.3412  ETA=148min
   ep10 [ 450/3337] loss=0.3397  ETA=146min
   ep10 [ 500/3337] loss=0.3414  ETA=143min
   ep10 [ 550/3337] loss=0.3431  ETA=141min
   ep10 [ 600/3337] loss=0.3414  ETA=138min
   ep10 [ 650/3337] loss=0.3406  ETA=136min
   ep10 [ 700/3337] loss=0.3402  ETA=133min
   ep10 [ 750/3337] loss=0.3398  ETA=131min
   ep10 [ 800/3337] loss=0.3411  ETA=128min
   ep10 [ 850/3337] loss=0.3403  ETA=125min
   ep10 [ 900/3337] loss=0.3405  ETA=123min
   ep10 [ 950/3337] loss=0.3403  ETA=120min
   ep10 [1000/3337] loss=0.3403  ETA=118min
   ep10 [1050/3337] loss=0.3401  ETA=115min
   ep10 [1100/3337] loss=0